In [84]:
import torchfrom torch import nnimport torchvisionimport torchvision.transforms as transformsfrom torch.utils.data import DataLoaderfrom torch_fidelity import calculate_metricsimport warningsimport osfrom torchvision.utils import save_imageimport matplotlib.pylab as pltfrom tqdm.auto import tqdmfrom timeit import default_timer as timerimport torch.nn.functional as Fimport numpy as npimport pandas as pdfrom sklearn.manifold import TSNEimport seaborn as snswarnings.filterwarnings('ignore')

## Device

In [85]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Timer

In [86]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## Data

In [87]:
transform = transforms.Compose([    transforms.ToTensor(),    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])# Download and load the training datatrainset = torchvision.datasets.CIFAR10(root='./data', train=True,                                        download=True, transform=transform)# Download and load the testing datatestset = torchvision.datasets.CIFAR10(root='./data', train=False,                                       download=True, transform=transform)

In [88]:
# DataLoadersbatch_size = 128 # batch size of the original paperdataloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

In [ ]:
class Generator(nn.Module):    def __init__(self, z_dim, channels_img, features_g, use_batch_norm=True, activation_type="relu"):        super(Generator, self).__init__()        if activation_type == "relu":            act_fn = nn.ReLU(True)        elif activation_type == "leaky_relu":            act_fn = nn.LeakyReLU(0.2, inplace=True)        elif activation_type == "tanh":            act_fn = nn.Tanh()        else:            raise ValueError(f"Unsupported activation type: {activation_type}")        self.gen = nn.Sequential(            # Layer 1            nn.ConvTranspose2d(z_dim, features_g * 4, kernel_size=4, stride=1, padding=0, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g * 4)] if use_batch_norm else []),            act_fn,            # Layer 2            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g * 2)] if use_batch_norm else []),            act_fn,            # Layer 3            nn.ConvTranspose2d(features_g * 2, features_g, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_g)] if use_batch_norm else []),            act_fn,            # Layer 4 (Output)            nn.ConvTranspose2d(features_g, channels_img, kernel_size=4, stride=2, padding=1, bias=True), # No BN on output            nn.Tanh(),        )    def forward(self, x):        return self.gen(x)

In [ ]:
class Discriminator(nn.Module):    def __init__(self, channels_img, features_d, use_batch_norm=True, activation_type="leaky_relu"):        super(Discriminator, self).__init__()        if activation_type == "relu":            act_fn = nn.ReLU(True)        elif activation_type == "leaky_relu":            act_fn = nn.LeakyReLU(0.2, inplace=True)        elif activation_type == "tanh":            act_fn = nn.Tanh()        else:            raise ValueError(f"Unsupported activation type: {activation_type}")        self.disc = nn.Sequential(            # Layer 1 (No BN on input like the paper)            nn.Conv2d(channels_img, features_d, kernel_size=4, stride=2, padding=1, bias=True),            act_fn,            # Layer 2            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_d * 2)] if use_batch_norm else []),            act_fn,            # Layer 3            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1, bias=not use_batch_norm),            *([nn.BatchNorm2d(features_d * 4)] if use_batch_norm else []),            act_fn,            # Layer 4 (Output)            nn.Conv2d(features_d * 4, 1, kernel_size=4, stride=1, padding=0, bias=True), # No BN on output            nn.Sigmoid(),        )    def forward(self, x):        return self.disc(x)

In [ ]:
def initialize_weights(model):    for m in model.modules():        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):            nn.init.normal_(m.weight.data, 0.0, 0.02)            if m.bias is not None:                 nn.init.constant_(m.bias.data, 0)        elif isinstance(m, nn.BatchNorm2d):            nn.init.normal_(m.weight.data, 1.0, 0.02)            nn.init.constant_(m.bias.data, 0)

In [ ]:
## Training loopdef train_dcgan(netD, netG, criterion, optimizerD, optimizerG, dataloader, device,                z_dim, num_epochs=5, fixed_noise_dim=64, sample_interval=100, show_result=True,                label_smooth_factor_real=0.0, # 0.0 means no smoothing                experiment_name="baseline"): # For saving images uniquely    start_time = timer()    fixed_noise = torch.randn(fixed_noise_dim, z_dim, 1, 1, device=device)    G_losses = []    D_losses = []    # a directory for this specific experiment's images    experiment_image_dir = f'generated_images_{experiment_name}'    os.makedirs(experiment_image_dir, exist_ok=True)    print(f"Starting DCGAN Training Loop for: {experiment_name}...")    for epoch in range(num_epochs):        for batch_idx, (real_images, _) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} ({experiment_name})", leave=False)):            real_images = real_images.to(device)            b_size = real_images.size(0)            # Labels for training            real_label_val = 1.0 - label_smooth_factor_real            fake_label_val = 0.0 # Standard: G tries to fool D into thinking its fakes are 0.0 (if D targets 0.0 for fake)                                 # or label_smooth_factor_real if you want to smooth fake labels too            ############################            # (1) Update D network            ###########################            netD.zero_grad()            label_real = torch.full((b_size,), real_label_val, dtype=torch.float, device=device)            output_real = netD(real_images).view(-1)            errD_real = criterion(output_real, label_real)            errD_real.backward()            D_x = output_real.mean().item()            noise = torch.randn(b_size, z_dim, 1, 1, device=device)            fake_images = netG(noise)            label_fake_for_D = torch.full((b_size,), fake_label_val, dtype=torch.float, device=device)            output_fake = netD(fake_images.detach()).view(-1)            errD_fake = criterion(output_fake, label_fake_for_D)            errD_fake.backward()            D_G_z1 = output_fake.mean().item()            errD = errD_real + errD_fake            optimizerD.step()            ############################            # (2) Update G network            ############################            netG.zero_grad()            # G wants D to think its fakes are "perfectly real" (label 1.0)            label_gen_real = torch.full((b_size,), 1.0, dtype=torch.float, device=device)            output_gen = netD(fake_images).view(-1)            errG = criterion(output_gen, label_gen_real)            errG.backward()            D_G_z2 = output_gen.mean().item()            optimizerG.step()            G_losses.append(errG.item())            D_losses.append(errD.item())            if show_result and batch_idx % sample_interval == 0:                print(f"Exp: {experiment_name} | Epoch [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(dataloader)}\t"                      f"Loss D: {errD.item():.4f}\tLoss G: {errG.item():.4f}\t"                      f"D(x): {D_x:.4f}\tD(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}")                with torch.no_grad():                    generated_samples = netG(fixed_noise).detach().cpu()                    normalized_generated_samples = (generated_samples + 1) / 2                    # Save to experiment-specific directory                    save_path = os.path.join(experiment_image_dir, f'epoch_{epoch+1}_batch_{batch_idx}.png')                    torchvision.utils.save_image(normalized_generated_samples, save_path, normalize=False)                    if batch_idx == 0 and epoch % 5 == 0 : # Show less frequently to avoid clutter                        plt.figure(figsize=(8, 8))                        plt.axis("off")                        plt.title(f"Generated Images: {experiment_name} - Epoch {epoch+1}, Batch {batch_idx}")                        plt.imshow(np.transpose(torchvision.utils.make_grid(normalized_generated_samples, padding=2, normalize=False).cpu(),(1,2,0)))                        plt.show()    end_time = timer()    print_train_time(start_time, end_time, device)    return G_losses, D_losses

In [ ]:
def evaluate_model(netG, z_dim_eval, device_eval, num_eval_images, batch_size_eval,                   real_images_dir, generated_images_base_dir, experiment_name_eval):    GENERATED_DIR_EXP = os.path.join(generated_images_base_dir, experiment_name_eval, "generated_for_eval")    os.makedirs(GENERATED_DIR_EXP, exist_ok=True)    print(f"Evaluating {experiment_name_eval}: Generating {num_eval_images} images...")    netG.eval()    img_idx = 0    with tqdm(total=num_eval_images, desc=f"Generating for {experiment_name_eval}") as pbar:        while img_idx < num_eval_images:            current_batch_size = min(batch_size_eval, num_eval_images - img_idx)            if current_batch_size <= 0: break            noise = torch.randn(current_batch_size, z_dim_eval, 1, 1, device=device_eval)            with torch.no_grad():                fake_images = netG(noise).detach().cpu()            for j in range(fake_images.size(0)):                if img_idx >= num_eval_images: break                save_image((fake_images[j] + 1) / 2, os.path.join(GENERATED_DIR_EXP, f"gen_img_{img_idx:05d}.png"))                img_idx += 1                pbar.update(1)    netG.train()    print(f"Calculating IS and FID for {experiment_name_eval}...")    # Calculate Inception Score on the generated images ONLY    isc_metrics = calculate_metrics(        input1=GENERATED_DIR_EXP, # The directory with generated images        isc=True,        fid=False, # Do not calculate FID here        verbose=False,        cuda=torch.cuda.is_available()    )    # Calculate Frechet Inception Distance between real and generated images    fid_metrics = calculate_metrics(        input1=real_images_dir,        input2=GENERATED_DIR_EXP,        isc=False, # Do not calculate IS here        fid=True,        verbose=False,        cuda=torch.cuda.is_available()    )    # Combine the results from the two calls    metrics = {**isc_metrics, **fid_metrics} # Merges the two dictionaries    print(f"Results for {experiment_name_eval}:")    print(f"  Inception Score (IS): mean={metrics['inception_score_mean']:.4f}, std={metrics['inception_score_std']:.4f}")    print(f"  Frechet Inception Distance (FID): {metrics['frechet_inception_distance']:.4f}")    return metrics# --- Prepare Real Images ONCE ---REAL_DIR_FOR_EVAL = "fid_images_real_cifar10"GENERATED_IMAGES_BASE_DIR = "all_generated_images_for_eval" # Parent dir for all experimentsos.makedirs(GENERATED_IMAGES_BASE_DIR, exist_ok=True)

In [ ]:
# --- Prepare Real Images ONCE for FID/IS Evaluation ---NUM_REAL_IMAGES_FOR_EVAL = 1000os.makedirs(REAL_DIR_FOR_EVAL, exist_ok=True)if len(os.listdir(REAL_DIR_FOR_EVAL)) < NUM_REAL_IMAGES_FOR_EVAL:    print(f"Populating {REAL_DIR_FOR_EVAL} with real CIFAR-10 images...")    # a temporary dataloader for saving real images without shuffling    real_img_save_loader = DataLoader(trainset, batch_size=100, shuffle=False)    saved_count = 0    for i, (images, _) in enumerate(tqdm(real_img_save_loader, desc="Saving real images")):        for j in range(images.size(0)):            if saved_count >= NUM_REAL_IMAGES_FOR_EVAL:                break            img_unnormalized = images[j] * 0.5 + 0.5            save_image(img_unnormalized, os.path.join(REAL_DIR_FOR_EVAL, f"real_img_{saved_count:05d}.png"))            saved_count += 1        if saved_count >= NUM_REAL_IMAGES_FOR_EVAL:            print(f"Saved {saved_count} real images to {REAL_DIR_FOR_EVAL}.")            break    if saved_count < NUM_REAL_IMAGES_FOR_EVAL:         print(f"Warning: Only saved {saved_count} real images, requested {NUM_REAL_IMAGES_FOR_EVAL}.")else:    print(f"Directory {REAL_DIR_FOR_EVAL} seems already populated with real images.")

Directory fid_images_real_cifar10 seems already populated with real images.
